In [1]:
# packages
import pandas as pd
from mod02_build_bot_predictor import train_model

### Define a function to extract predictions from the model

In [2]:
def predict_bot(df, model=None):
    """
    Predict whether each account is a bot (1) or human (0).
    """
    if model is None:
        model = train_model()

    preds = model.predict(df)
    return pd.Series(preds, index=df.index)

### Define a function to evaluate model error

In [3]:
def confusion_matrix_and_metrics(y_true, y_pred):
    """
    Computes confusion matrix and common error rates for binary classification.

    Assumes labels:
      0 = negative class
      1 = positive class

    Returns:
      dict with:
        tn, fp, fn, tp
        misclassification_rate
        false_positive_rate
        false_negative_rate
    """
    tn = fp = fn = tp = 0

    for yt, yp in zip(y_true, y_pred):
        if yt == 0 and yp == 0:
            tn += 1
        elif yt == 0 and yp == 1:
            fp += 1
        elif yt == 1 and yp == 0:
            fn += 1
        elif yt == 1 and yp == 1:
            tp += 1
        else:
            raise ValueError("Labels must be 0 or 1")

    total = tn + fp + fn + tp

    misclassification_rate = (fp + fn) / total if total > 0 else 0.0
    false_positive_rate = fp / (fp + tn) if (fp + tn) > 0 else 0.0
    false_negative_rate = fn / (fn + tp) if (fn + tp) > 0 else 0.0

    return {
        "tp": tp,
        "tn": tn,
        "fp": fp,
        "fn": fn,
        "misclassification_rate": misclassification_rate,
        "false_positive_rate": false_positive_rate,
        "false_negative_rate": false_negative_rate,
    }


### Load the data

In [4]:
TRAIN_PATH = "mod02_data/train.csv"
train = pd.read_csv(TRAIN_PATH)

TEST_PATH = "mod02_data/test.csv"
test = pd.read_csv(TEST_PATH)

### Format the data by independent vs. dependent variables

In [5]:
X_train = train.drop(columns=["is_bot"])
y_train = train['is_bot']

X_test = test.drop(columns=["is_bot"])
y_test = test['is_bot']

### Build the model on training data

In [6]:
model = train_model(X_train, y_train)

### Get the model predictions on training and test data

In [7]:
y_pred_train = predict_bot(X_train, model)
y_pred_test = predict_bot(X_test, model)

### Check results on the training set (data used to build the model)

In [8]:
confusion_matrix_and_metrics(y_train, y_pred_train)

{'tp': 121,
 'tn': 2586,
 'fp': 51,
 'fn': 242,
 'misclassification_rate': 0.09766666666666667,
 'false_positive_rate': 0.019340159271899887,
 'false_negative_rate': 0.6666666666666666}

### Check results on the test set (new data not yet seen by the model)

In [9]:
confusion_matrix_and_metrics(y_test, y_pred_test)

{'tp': 34,
 'tn': 857,
 'fp': 17,
 'fn': 92,
 'misclassification_rate': 0.109,
 'false_positive_rate': 0.019450800915331808,
 'false_negative_rate': 0.7301587301587301}

# Discussion Questions

### Based on the misclassification rate of your model, discuss your confidence in the ability to predict a bot. 

My confidence in the ability to predict a bot is semi low. This is due to only being able to get the misclassification_rate to output at .103. So a little over 10% of the time the model is misclassifying. If I was able to get the percentage down more I would have a much higher confidence in being able to predict a bot.

### What are potential ramifications of false positives from the model?

I feel like the main purpose of models that detect bots is to make sure that only real people get into websites. The main ramification of a false positive is that real humans could be locked out or potentiall banned from websites for simply trying to enter. This could lose a company revenue for example if real humans could not access their website.

### What are potential ramifications of false negatives from the model?

The main purpose of bot predictors when it comes to websites is to block bots. I have recently seen this in effect when there is a limited supply of a product online and retailers will block bots so that real people have a chance at buying the item. A ramification of a false negative in this instance would allow bots in and the bots could buy all of the stock to then later be resold at a higher value. This could damage a companies reputation and have customers lose trust.